<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L05-leakage-harness/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L05-leakage-harness/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/proteomics-multiomics/lessons/P07-L05-leakage-harness/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/proteomics-multiomics/lessons/P07-L05-leakage-harness/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P07-L05 · Cross-modality QC and leakage traps: a harness that refuses to report a leaky biomarker

**You will build:** the checks that stand between a paired proteomics and metabolomics cohort and a
biomarker report: a finder for samples registered twice under two donor ids, a merge that turns them into
one real donor, a count of cases per batch that counts donors and not samples, a label permutation that
shuffles donors and not samples, and a gate that refuses to return an AUC while any check fails. Then you
will **measure**, against an answer key, what each trap adds to a cross-validated biomarker AUC and which
check catches which trap.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download · **Prerequisites:**
`P07-L04-factor-models-two-omics` (a matched two-layer cohort with a planted technical factor and an
answer key) and `T01-L02-leakage-clinic` (the five-leak taxonomy this lesson extends, not repeats). Python
and numpy, including `np.unique` with `return_index` and `return_inverse`.

By the end you will be able to:

1. Implement a duplicate-sample finder that compares profiles across modalities, and a merge of the donors it
   links, so that a split keeps one real person's samples together.
2. Implement a case count per batch that counts donors, and a label permutation that shuffles whole units,
   optionally inside a group; together they give a permutation p-value for batch confounding.
3. Implement a release gate that returns a biomarker AUC only when no check fails, and names the failing
   checks in a fixed order otherwise.
4. Measure, over a set of simulated cohorts, how much cross-validated AUC each trap adds (donors on both sides of
   a split, duplicate samples, a batch confounded with the outcome) against a donor-grouped baseline.
5. Explain, from the printed negative-control table, which control catches which trap and which is blind to it.

**The data is synthetic, and the simulator is in this notebook.** A real cohort has no answer key, so it can
never tell you which of its samples are duplicates or whether a biomarker was a batch. A simulated one does.
`meta.yaml` declares the data as synthetic and nothing is downloaded. Nothing you conclude about these
numbers describes a real cohort: what transfers is the method, and the habit of asking every protocol what it
reports when the labels are shuffled.

**What this lesson does not do.** A feature that is derived from the outcome is T01-L02's target leak and is not
repeated here. There is one model family and one family of splits, and the checks are tuned to a world built
for them. The sizes printed below are what these worlds show, not what a real study would.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `P07-L04`, `T01-L02`

**This lesson assumes you can already:**

- write Python functions with loops and dicts, and index a numpy array with an integer array or a boolean mask
- say what AUC measures, as the chance that a randomly chosen case is scored above a randomly chosen control
- describe k-fold cross-validation, and say why the samples of one donor are not independent of each other (T01-L02 builds this)
- use np.unique with return_index and return_inverse, and np.einsum or a matrix product for a table of dot products

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Cross-validation: evaluating estimator performance (iterators for grouped data; permutation test score)](https://scikit-learn.org/stable/modules/cross_validation.html) · scikit-learn developers · *free* — GroupKFold and LeaveOneGroupOut are the donor-grouped and leave-one-plate-out folds used here; its permutation test score is the sample-level shuffle that exercise 4 replaces with a shuffle between donors
- [The need to approximate the use-case in clinical machine learning](https://pmc.ncbi.nlm.nih.gov/articles/PMC5441397/) · Saeb, Lonini, Jayaraman, Mohr and Kording (GigaScience 2017, open access) · *free* — record-wise against subject-wise cross-validation when each person gives several samples, which is the donor trap of section 2 and the reason folds must keep donors whole
- [Tackling the widespread and critical impact of batch effects in high-throughput data](https://pmc.ncbi.nlm.nih.gov/articles/PMC3880143/) · Leek et al. (Nature Reviews Genetics 2010; author manuscript in PubMed Central) · *free* — why a batch that is correlated with the outcome is a major problem, the hazard behind the case share per plate of exercise 3 and the batch p-value
- [Permutation Tests for Studying Classifier Performance](https://www.jmlr.org/papers/v11/ojala10a.html) · Ojala and Garriga (Journal of Machine Learning Research 2010) · *free* — the permutation test that asks whether a classifier has found real class structure, by permuting the labels, which is the idea behind exercise 4 and the negative-control table
- [numpy.unique](https://numpy.org/doc/stable/reference/generated/numpy.unique.html) · NumPy documentation · *free* — the return_index and return_inverse options that give one entry per donor or unit and the way back to one entry per sample, used in exercises 3 and 4

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import math
import sys
import traceback
from typing import Callable, NamedTuple

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

# The world. Every number that shapes it is here, so that nothing later is a hidden knob.
SEED = 20261019              # this lesson's one seed: every world derives from it, so re-running prints the same
                             # numbers anywhere
N_WORLDS = 8                 # independently generated worlds; every table below averages over them
N_DONORS = 80                # donors per world, before any is registered twice
N_CASES = 40                 # ... of whom this many are cases
MODALITIES = ("protein", "metabolite")
N_FEATURES = {"protein": 150, "metabolite": 100}
N_TRUE = {"protein": 12, "metabolite": 8}       # features that really differ between cases and controls
N_BATCHES = {"protein": 4, "metabolite": 3}     # plates / runs per modality
CASE_SHIFT = 0.75            # the true difference, in units of the noise on one feature
BATCH_SD, DONOR_SD, BIO_SD, TECH_SD = 1.2, 0.9, 0.95, 0.3   # spread of the batch, donor, biological and technical parts
N_DUP_DONORS = 12            # donors whose samples are registered a second time under a new id (duplicates cohort)
CONFOUND = 0.8               # in the confounded cohort: chance that a case lands on a case-heavy protein plate

# The analysis settings, fixed for the whole lesson so that only the choices you study vary.
TOP_K = 10                   # features kept by the model, chosen inside each training fold
NN_K = 5                     # neighbours that vote
CV_FOLDS = 5
R_MIN = 0.9                  # two samples this similar (or more) under different donor ids are flagged
N_PERM_NULL = 20             # label permutations behind each negative-control AUC
N_PERM_BATCH = 199           # label permutations behind each batch p-value
BATCH_ALPHA = 0.01           # a batch p-value at or below this is "confounded"
NULL_MARGIN = 0.05           # a negative-control AUC this far above chance (0.5) is an alarm

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("duplicate_pairs",),
    "exercise 2": ("merge_donors",),
    "exercise 3": ("batch_case_share",),
    "exercise 4": ("permute_units",),
    "exercise 5": ("release_gate",),
}
# The demos (the tables below the exercises) run when this file is the program being run, that is in a
# notebook or as `python lesson.py`; the autograder imports the file and only needs the checks.
_DEMOS = __name__ == "__main__"


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (batch_case_share)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _signed(x: float) -> str:
    """+0.012 or -0.012, and +0.000 rather than -0.000 for a value that rounds to zero."""
    v = round(float(x), 3)
    return f"{(v if v != 0 else 0.0):+.3f}"


def _se(values) -> float:
    """Standard error of the mean of a list of numbers (0.0 for fewer than two)."""
    v = np.asarray(values, dtype=float)
    return float(v.std(ddof=1) / math.sqrt(len(v))) if len(v) > 1 else 0.0


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 60     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 1024     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _try_rows(label: str, header: str, rows: list, footer: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a table whose rows are slow as one `_try` per row, so that the time limit covers one row and not the whole table.

    `rows` is a list of functions that each print one row and `footer` prints what follows them. If an exercise the table
    `needs` has not passed, the whole table is skipped with one line that names it, as `_try` does for any demo.
    """
    if any(_STATUS.get(name) != "passed" for name in needs):
        _try(label, footer, needs)         # prints the one line that names what the table is waiting for
        return
    print(header)
    for k, row in enumerate(rows, 1):
        _try(f"{label}, row {k}", row)
    _try(f"{label}, summary", footer)

## 1. A paired cohort with three traps planted

A study recruits donors, some of them cases, and takes several samples from each. Every sample is measured
twice over: a proteomics-style layer and a metabolomics-style layer, on a log scale and already normalised.
A value is the feature's usual level, plus the plate or run the sample went through, plus a persistent donor
effect, plus a small shift for cases on a few true features, plus biological and technical noise.
Cases are followed up more often than controls, on purpose; the cell below prints how many samples each donor
gives.

**What an honest evaluation has to imitate.** A biomarker will be used on *new* donors, measured on plates that
do not sort people by diagnosis (a clinic does not put its patients on a plate by their disease). So a protocol is
honest when its test samples come from donors the model has never seen, and when nothing about the plate tells it
the answer. (Kapoor and Narayanan's taxonomy of leakage lists non-independence between training and test samples
as leakage unless the claim is about a distribution with the same dependence structure, and a claim about newly
recruited donors is not one; `claims.yaml`.) Three traps break that, each planted somewhere else:

| trap | what is planted | who is fooled |
|---|---|---|
| donor | every donor carries a persistent signature in both layers | a split that puts a donor on both sides lets a flexible model recognise the donor |
| duplicate | some donors are registered twice under a new id, every sample measured again on the same plates | a split that keeps donors whole *by the manifest* still separates the copies |
| batch | in one cohort the protein plates hold mostly cases or mostly controls; the metabolite runs stay balanced | every donor-grouped split of that cohort: the plate predicts status for new donors too |

`make_cohort` builds a cohort in four flavours from one seed: **balanced** (plates balanced by status, no
duplicates), **duplicates**, **confounded**, and **all three**. The flavours of one world share their donors and
their biology and differ only in the trap, so a difference between two of them is the trap and nothing else.
The cohort also keeps an **answer key** (`identity`, `copy_of`), so that this notebook can grade your checks.
The checks you write never read it. Read the cell below; nothing in it is graded.

In [ ]:
class Cohort(NamedTuple):
    """One generated cohort. Row i of every array describes the same sample."""
    protein: np.ndarray            # (samples, features) log-intensity-like values
    metabolite: np.ndarray         # (samples, features) log-abundance-like values
    donor: np.ndarray              # the donor id in the MANIFEST; a re-registered sample carries a fresh id
    y: np.ndarray                  # 1 for a sample of a case, 0 for a sample of a control
    batch_protein: np.ndarray      # the plate each sample's protein values came from
    batch_metabolite: np.ndarray   # the run each sample's metabolite values came from
    identity: np.ndarray           # ANSWER KEY: the real donor behind each sample (copies point at their original)
    copy_of: np.ndarray            # ANSWER KEY: the row of the sample a re-registered sample copies, else -1


def make_cohort(world: int, duplicates: bool = False, confounded: bool = False) -> Cohort:
    """Generate one cohort of world number `world`. Given.

    Donors are cases or controls (a fixed split). Each donor runs all of its samples on one plate per layer.
    With `confounded`, a case lands on a low protein plate (the first half) with chance CONFOUND and a control
    on a high one with the same chance; otherwise plates are filled in blocks, so every plate holds equally many
    cases and controls. The metabolite runs are always filled in blocks. With `duplicates`, N_DUP_DONORS donors
    have every sample registered a second time under a fresh donor id, measured again (same plate, same material,
    new technical noise). The arrays of a cohort are read-only: they are shared by every table, so write to copies.
    """
    def stream(tag: int) -> np.random.Generator:       # one stream per ingredient, so flavours share their biology
        return np.random.default_rng([SEED, world, tag])

    r = stream(1)
    is_case = (r.permutation(N_DONORS) < N_CASES).astype(np.int64)
    visits = np.where(is_case == 1, r.integers(3, 5, N_DONORS), r.integers(2, 4, N_DONORS))
    donor = np.repeat(np.arange(N_DONORS), visits)
    y = is_case[donor]
    n = len(donor)
    layers, plates, in_tube = {}, {}, {}
    for k, name in enumerate(MODALITIES):
        p, n_plates = N_FEATURES[name], N_BATCHES[name]
        r = stream(10 + k)
        level = 8.0 + 2.0 * r.standard_normal(p)
        donor_effect = DONOR_SD * r.standard_normal((N_DONORS, p))
        plate_effect = BATCH_SD * r.standard_normal((n_plates, p))
        true_features = r.permutation(p)[:N_TRUE[name]]
        true_sign = np.where(r.random(N_TRUE[name]) < 0.5, -1.0, 1.0)
        r = stream(20 + k)
        if confounded and name == "protein":
            half = n_plates // 2
            with_own_class = r.random(N_DONORS) < CONFOUND
            low, high = r.integers(0, half, N_DONORS), r.integers(half, n_plates, N_DONORS)
            plate_of = np.where(is_case == 1, np.where(with_own_class, low, high),
                                np.where(with_own_class, high, low))
        else:
            plate_of = np.empty(N_DONORS, dtype=np.int64)
            for status in (0, 1):
                members = r.permutation(np.flatnonzero(is_case == status))
                plate_of[members] = np.arange(len(members)) % n_plates
        shift = np.zeros((N_DONORS, p))
        shift[:, true_features] = CASE_SHIFT * is_case[:, None] * true_sign[None, :]
        material = (level + plate_effect[plate_of[donor]] + donor_effect[donor] + shift[donor]
                    + BIO_SD * stream(30 + k).standard_normal((n, p)))
        in_tube[name] = material
        layers[name] = material + TECH_SD * stream(40 + k).standard_normal((n, p))
        plates[name] = plate_of[donor]
    identity = donor.copy()
    copy_of = np.full(n, -1, dtype=np.int64)
    if duplicates:
        copied = np.sort(stream(50).permutation(N_DONORS)[:N_DUP_DONORS])
        rows = np.flatnonzero(np.isin(donor, copied))
        fresh_id = N_DONORS + np.searchsorted(copied, donor[rows])
        for k, name in enumerate(MODALITIES):
            again = TECH_SD * stream(60 + k).standard_normal((len(rows), N_FEATURES[name]))
            layers[name] = np.vstack([layers[name], in_tube[name][rows] + again])
            plates[name] = np.concatenate([plates[name], plates[name][rows]])
        y = np.concatenate([y, y[rows]])
        copy_of = np.concatenate([copy_of, rows])
        identity = np.concatenate([identity, donor[rows]])
        donor = np.concatenate([donor, fresh_id])
    cohort = Cohort(layers["protein"], layers["metabolite"], donor, y, plates["protein"], plates["metabolite"],
                    identity, copy_of)
    for array in cohort:
        array.setflags(write=False)        # every table reuses these cohorts: an edit in place must fail loudly, not corrupt them
    return cohort


COHORT_KINDS = {"balanced": (False, False), "duplicates": (True, False),
                "confounded": (False, True), "all three": (True, True)}
_COHORTS: dict = {}


def cohort_of(world: int, kind: str) -> Cohort:
    """The cohort of flavour `kind` (a key of COHORT_KINDS) in world `world`, built once. Given."""
    if (world, kind) not in _COHORTS:
        _COHORTS[(world, kind)] = make_cohort(world, *COHORT_KINDS[kind])
    return _COHORTS[(world, kind)]


if _DEMOS:
    c = cohort_of(0, "balanced")
    per_donor = np.bincount(c.donor)
    cases = np.flatnonzero(np.bincount(c.donor, weights=c.y) > 0)
    d = cohort_of(0, "duplicates")
    print(f"world 0, balanced: {len(c.y)} samples from {len(per_donor)} donors ({len(cases)} cases); "
          f"{c.protein.shape[1]} protein features, {c.metabolite.shape[1]} metabolite features")
    print(f"samples per donor: cases {per_donor[cases].min()} to {per_donor[cases].max()}, "
          f"controls {np.delete(per_donor, cases).min()} to {np.delete(per_donor, cases).max()}; "
          f"{int(c.y.sum())} of {len(c.y)} samples belong to a case")
    print(f"plates: {len(np.unique(c.batch_protein))} protein, {len(np.unique(c.batch_metabolite))} metabolite")
    print(f"world 0, duplicates: {len(d.y)} samples under {len(np.unique(d.donor))} manifest donor ids; "
          f"the answer key says {len(np.unique(d.identity))} real donors and {int((d.copy_of >= 0).sum())} "
          f"samples that copy another")

## 2. A biomarker model, four ways to split, and what each trap adds

The model is deliberately the kind that can memorise. Inside the training samples only, it ranks every
feature by the absolute difference between cases and controls (a Welch-type t statistic), keeps the top
`TOP_K`, standardises them, and scores a test sample by an inverse-distance-weighted vote of its `NN_K` nearest
training samples. Choosing the features inside the training fold is the honest way to do it. Note what it
still ignores: the t statistic treats every sample as independent, although samples of one donor are not.
The table below shows that this model memorises donors. No other model was measured, so the lesson says nothing
about how much any other model would leak.

A **protocol** is a way of cutting the samples into training and test folds:

* **random rows**: any sample can land on either side, so a donor's samples are spread over both;
* **donors whole**: every manifest donor id lies wholly in one fold (`group_folds`, as in T01-L02);
* **real donors whole**: the same, using the answer key's identity, which a real study does not have;
* **leave one plate out**: each fold tests on one plate and trains on the others.

Random rows against donors whole is the distinction Saeb and colleagues call record-wise against subject-wise
cross-validation; they found record-wise cross-validation often massively overestimates accuracy
(`claims.yaml`). In scikit-learn (1.9.1 documentation, read on 19 October 2026) `GroupKFold` keeps every group in one
test fold and `LeaveOneGroupOut` is the leave-one-plate-out above. This lesson writes its own, so it needs only numpy.

The cell below is given (AUC by ranks, the model, three fold builders and the function that picks one). Run the one after it: it measures
the cross-validated AUC of every protocol in `PROTOCOLS` over `N_WORLDS` worlds. Every difference is **paired by
world**: the two cohorts compared share their donors and their biology, so a difference is the trap and
nothing else. Read it carefully, because every later section is about catching what this table shows.

In [ ]:
def auc_score(y_true, scores) -> float:
    """Area under the ROC curve by the rank (Mann-Whitney) identity, ties given the average rank. Given."""
    y = np.asarray(y_true)
    s = np.asarray(scores, dtype=float)
    n_pos = int(y.sum())
    n_neg = int(y.size - n_pos)
    if n_pos == 0 or n_neg == 0:
        raise ValueError("AUC is undefined when one class is absent")
    order = np.argsort(s, kind="mergesort")
    _, inverse, counts = np.unique(s[order], return_inverse=True, return_counts=True)
    ends = np.cumsum(counts)
    average_rank = (ends - counts + 1 + ends) / 2.0
    ranks = np.empty(s.size)
    ranks[order] = average_rank[inverse]
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def biomarker_scores(X: np.ndarray, y: np.ndarray, train, test) -> np.ndarray:
    """Score the `test` samples with a model fitted on the `train` samples only. Given.

    Inside the training samples: rank the features by |t| between cases and controls, keep TOP_K, standardise
    them, and score each test sample by the inverse-distance-weighted label of its NN_K nearest training samples.
    """
    tr, te = np.asarray(train), np.asarray(test)
    Xt, yt = X[tr], y[tr]
    cases, controls = Xt[yt == 1], Xt[yt == 0]
    if len(cases) < 2 or len(controls) < 2:
        raise ValueError("the training samples need at least two cases and two controls")
    spread = cases.var(axis=0, ddof=1) / len(cases) + controls.var(axis=0, ddof=1) / len(controls)
    t = (cases.mean(axis=0) - controls.mean(axis=0)) / np.sqrt(spread + 1e-12)
    top = np.argsort(-np.abs(t), kind="mergesort")[:TOP_K]
    centre, scale = Xt[:, top].mean(axis=0), Xt[:, top].std(axis=0) + 1e-12
    z_train, z_test = (Xt[:, top] - centre) / scale, (X[te][:, top] - centre) / scale
    squared = ((z_test[:, None, :] - z_train[None, :, :]) ** 2).sum(axis=2)
    near = np.argsort(squared, axis=1, kind="mergesort")[:, :NN_K]
    weight = 1.0 / (np.take_along_axis(squared, near, axis=1) + 1e-6)
    return (weight * yt[near]).sum(axis=1) / weight.sum(axis=1)


def random_folds(n_rows: int, k: int, seed: int) -> list:
    """k folds of randomly chosen rows. Given. Returns [(train_idx, test_idx), ...]."""
    order = np.random.default_rng([SEED, 1, seed]).permutation(n_rows)
    fold = np.empty(n_rows, dtype=np.int64)
    fold[order] = np.arange(n_rows) % k
    return [(np.flatnonzero(fold != f), np.flatnonzero(fold == f)) for f in range(k)]


def group_folds(groups: np.ndarray, k: int, seed: int) -> list:
    """k folds in which every group (a donor, or a unit) lies wholly in one fold. Given."""
    ids = np.unique(groups)
    order = np.random.default_rng([SEED, 2, seed]).permutation(len(ids))
    fold_of = np.empty(len(ids), dtype=np.int64)
    fold_of[order] = np.arange(len(ids)) % k
    fold = fold_of[np.searchsorted(ids, groups)]
    return [(np.flatnonzero(fold != f), np.flatnonzero(fold == f)) for f in range(k)]


def batch_folds(batch: np.ndarray) -> list:
    """One fold per batch: train on every other batch, test on this one. Given."""
    return [(np.flatnonzero(batch != b), np.flatnonzero(batch == b)) for b in np.unique(batch)]


def cv_auc(X: np.ndarray, y: np.ndarray, folds: list) -> float:
    """Mean over folds of the AUC of the fold's test samples; a fold with one class only is skipped. Given."""
    values = []
    for train, test in folds:
        labels = y[test]
        if 0 < labels.sum() < len(labels):
            values.append(auc_score(labels, biomarker_scores(X, y, train, test)))
    if not values:
        raise ValueError("no fold has both classes")
    return float(np.mean(values))


def panel_matrix(cohort: Cohort, panel: tuple) -> np.ndarray:
    """The feature table of a panel: the chosen layers side by side. Given."""
    return np.hstack([getattr(cohort, name) for name in panel])


def folds_for(cohort: Cohort, how: str, world: int) -> list:
    """The folds of a protocol: "rows", "donor" (manifest ids), "identity" (answer key) or "plate" (protein). Given."""
    if how == "rows":
        return random_folds(len(cohort.y), CV_FOLDS, world)
    if how == "donor":
        return group_folds(cohort.donor, CV_FOLDS, world)
    if how == "identity":
        return group_folds(cohort.identity, CV_FOLDS, world)
    if how == "plate":
        return batch_folds(cohort.batch_protein)
    raise ValueError(f"unknown protocol {how!r}")


PROTOCOLS = {   # key: (what it is, cohort flavour, panel, folds)
    "A": ("balanced · both layers · donors whole", "balanced", MODALITIES, "donor"),
    "B": ("balanced · both layers · random rows", "balanced", MODALITIES, "rows"),
    "C": ("duplicates · both layers · manifest donors whole", "duplicates", MODALITIES, "donor"),
    "D": ("duplicates · both layers · real donors whole", "duplicates", MODALITIES, "identity"),
    "E": ("balanced · protein only · donors whole", "balanced", ("protein",), "donor"),
    "F": ("confounded · protein only · donors whole", "confounded", ("protein",), "donor"),
    "G": ("confounded · protein only · leave one plate out", "confounded", ("protein",), "plate"),
    "H": ("all three traps · both layers · random rows", "all three", MODALITIES, "rows"),
}
AUC_BY: dict = {}


def protocol_auc(key: str, world: int) -> float:
    """Cross-validated AUC of protocol `key` in world `world`, remembered. Given."""
    if (key, world) not in AUC_BY:
        _, kind, panel, how = PROTOCOLS[key]
        cohort = cohort_of(world, kind)
        AUC_BY[(key, world)] = cv_auc(panel_matrix(cohort, panel), cohort.y, folds_for(cohort, how, world))
    return AUC_BY[(key, world)]


def _paired(first: str, second: str) -> np.ndarray:
    """Per-world AUC of protocol `first` minus protocol `second`. Given."""
    return np.array([protocol_auc(first, w) - protocol_auc(second, w) for w in range(N_WORLDS)])


def _auc_table() -> None:
    print(f"cross-validated AUC, mean over {N_WORLDS} worlds (standard error in brackets)")
    for key, (what, *_rest) in PROTOCOLS.items():
        values = [protocol_auc(key, w) for w in range(N_WORLDS)]
        print(f"  {key}  {what:<52} {np.mean(values):.3f}  ({_se(values):.3f})")
    print("\npaired differences, protocol minus protocol, same worlds ('clearly' is a rule of thumb here: more than two standard errors from zero):")
    for name, (a, b) in (("donors on both sides of a split", ("B", "A")),
                         ("duplicate samples split apart", ("C", "D")),
                         ("batch confounded with outcome", ("F", "E")),
                         ("... and what leaving a plate out removes", ("F", "G")),
                         ("... and where leaving a plate out lands", ("G", "E")),
                         ("all three together, random rows", ("H", "A"))):
        diff = _paired(a, b)
        verdict = ("above zero" if diff.mean() > 2 * _se(diff)
                   else "below zero" if diff.mean() < -2 * _se(diff) else "not clearly different from zero")
        print(f"  {a} minus {b}: {name:<42} {_signed(diff.mean())}  (standard error {_se(diff):.3f}) -> {verdict}")
    leak, removed, lands = _paired("F", "E"), _paired("F", "G"), _paired("G", "E")
    side = ("below" if lands.mean() < -2 * _se(lands) else "above" if lands.mean() > 2 * _se(lands)
            else "not clearly different from")
    print(f"\nleaving a plate out of the confounded cohort changed its score by {_signed(-removed.mean())}, against a plate leak of "
          f"{_signed(leak.mean())}, and leaves it {side} the balanced reference E ({_signed(lands.mean())}); that split asks "
          "how well the model scores a plate it has not seen, which is a different question from E's")


if _DEMOS:
    _auc_table()

**Reading the table.** Protocol A is the reference: the AUC a biomarker earns on new donors when nothing is
planted against it. Each paired difference is what one trap adds to a score that then goes into a paper. None
of the three needs a bad model; each is a property of the cohort or of the split, and each can hide inside a
number that looks like an ordinary cross-validated AUC. The rest of the lesson builds the checks that find
them *without* an answer key, and a gate that refuses to report a biomarker while a check fails. Where the
table says a difference is "not clearly different from zero", it means exactly that for these worlds and this
model, not that the trap is harmless.

The rows and the line about leaving a plate out ask "can a better split cure a confounded plate?". F minus G is how
much of the confounded score disappears when the test plate is never in training, G minus E is where the score then
lands against the balanced reference E, and the line under the table compares both with the plate leak. Read what they
print for these worlds; whatever it is, that split answers a different question (how well does the model score a
plate it has never seen?), so it does not turn the confounded cohort into a balanced one.

## 3. Exercise 1 — `duplicate_pairs()`: the same material under two donor ids

T01-L02 found duplicate rows by comparing keys; Kapoor and Narayanan list duplicates in datasets as a type of
leakage (`claims.yaml`). A measurement is never a key: a sample measured twice differs from itself by technical
noise in every feature, so compare **profiles**. For one layer, put the features on a
common scale over the samples (centre each feature on its mean, divide it by its population standard deviation,
`ddof=0`; a feature with no spread becomes zeros), then call two samples **similar** when the cosine of their two
rows is high. Without the centring every pair of log intensities looks alike, because the feature means dominate.

Report a pair when the manifest gives its two samples **different** donor ids and the similarity is **strictly
above** `r_min` in **at least one** layer: a copy of the material leaks into any model that uses even one layer.
Compare the similarity itself with `r_min`: nothing is rounded and no tolerance is added, so a pair a hair above
the threshold is reported and one exactly on it is not. A row that is all zeros after scaling has similarity 0.0
with every sample. Return the pairs `(i, j)` with `i < j` as a sorted list of tuples of plain ints. Do not change
`blocks` or `donor`: the cohorts are reused by every table below, so work on copies.

<details><summary>💡 Hint 1 — what to think about</summary>

What is the cosine of two raw log-intensity rows before any centring? Along which axis must `mean` and `std` run so
that it is each *feature* that is put on a common scale, and what do you do with a feature whose spread is 0?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For each layer: scale the columns, shrink every row to length one (leave an all-zero row alone), and take every
row-by-row dot product at once. Keep one yes/no table of pairs and combine the layers with "or". Keep only the
upper triangle and only the pairs whose donor ids differ, then turn the positions that are left into tuples.
</details>

In [ ]:
def duplicate_pairs(blocks: list, donor: np.ndarray, r_min: float) -> list:
    """Pairs of samples with different donor ids whose profiles are more similar than `r_min` in some layer.

    `blocks` is a list of 2-D float arrays (samples x features), one per layer, with the same samples in the
    same order. Per layer: centre every feature on its mean over the samples and divide it by its population
    standard deviation (a feature with zero standard deviation becomes all zeros); the similarity of samples
    i and j is the cosine of their two scaled rows (0.0 if either row is all zeros). Return the pairs (i, j)
    with i < j, donor[i] != donor[j] and similarity > r_min (strictly, nothing rounded) in at least one layer.
    Do not change `blocks` or `donor` (work on copies).

    Example:
        >>> a = np.array([[1.0, 2.0, 0.0, 5.0], [4.0, 1.0, 3.0, 2.0], [1.1, 2.1, 0.1, 5.1], [0.0, 5.0, 2.0, 1.0]])
        >>> duplicate_pairs([a], [10, 11, 12, 13], 0.9)
        [(0, 2)]
        >>> duplicate_pairs([a], [10, 11, 10, 13], 0.9)       # samples 0 and 2 share a donor id: not a pair
        []

    Returns:
        A sorted list of (int, int) tuples.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_duplicate_pairs() -> None:
    b = np.array([[1.0, 2.0, 0.5, -1.0], [-2.0, 0.5, 1.0, 1.5], [0.3, -1.0, 2.0, 0.2],
                  [1.02, 1.98, 0.52, -1.01], [-0.5, 1.5, -1.0, 0.0]])
    donors = [0, 1, 2, 3, 4]
    own, own_donor = b.copy(), np.array(donors)
    duplicate_pairs([own], own_donor, 0.9)
    assert np.array_equal(own, b) and np.array_equal(own_donor, donors), (
        "do not change the arrays you are given: work on copies (centring a block in place alters the cohort that every "
        "later table reuses)")
    got = duplicate_pairs([b + 100.0], donors, 0.9)
    assert all(type(a) is int and type(b) is int for a, b in got), (
        "return plain Python ints in each (i, j) tuple (int(i), int(j)), not numpy integers")
    assert got == [(0, 3)], (
        f"expected [(0, 3)], got {got}. Centre and scale EACH FEATURE over the samples before the cosine: a large "
        "constant added to every value (here 100) must change nothing, and raw log intensities all look alike. "
        "Return each pair once, with i < j")
    assert duplicate_pairs([b], [0, 1, 2, 0, 4], 0.9) == [], (
        "samples 0 and 3 carry the SAME donor id here: they are the same donor's repeat, not a mislabelled duplicate")
    unlike = np.array([[2.0, -1.0, 0.0], [-1.0, 3.0, 1.0], [0.0, 0.5, -2.0], [1.5, 1.0, 2.5], [-2.5, -1.0, 1.0]])
    only_second = duplicate_pairs([unlike, b], donors, 0.9)
    assert only_second == [(0, 3)], (
        f"a pair that is similar in only the SECOND layer must be reported (at least one layer), got {only_second}")
    assert duplicate_pairs([b], donors, 0.9999999) == [] and duplicate_pairs([b], donors, 0.99) == [(0, 3)], (
        "the similarity must be compared with r_min (strictly above), not with a fixed number")
    ones = np.tile([[1.0], [2.0], [1.0], [2.0]], (1, 3))
    assert duplicate_pairs([ones, ones * 0.0 + 5.0], [0, 1, 2, 3], 0.5) == [(0, 2), (1, 3)], (
        "identical rows in a layer are similar; a layer whose features are all constant must give zeros, not NaN")
    assert duplicate_pairs([np.hstack([b, np.full((5, 1), 7.0)])], donors, 0.9) == [(0, 3)], (
        "a constant feature next to real ones must give zeros, not NaN, so the other features still decide")
    assert duplicate_pairs([b], [4, 3, 2, 1, 0], 0.9) == [(0, 3)], (
        "compare donor ids for DIFFERENCE (!=), whichever id is larger")
    big = b.copy()
    big[:, 1] *= 1000.0
    assert duplicate_pairs([big], donors, 0.9) == [(0, 3)], (
        "multiplying one feature by a constant must change nothing: divide every feature by its own standard deviation")
    tie = np.array([[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, -1.0], [-1.0, -1.0, -1.0, 1.0], [-1.0, -1.0, -1.0, -1.0]])
    assert duplicate_pairs([tie], [0, 1, 2, 3], 0.5) == [] and \
        duplicate_pairs([tie], [0, 1, 2, 3], 0.499999) == [(0, 1), (2, 3)], (
        "rows 0-1 and 2-3 have similarity exactly 0.5 (every column is balanced): a pair is reported only when its "
        "similarity is strictly above r_min, so none at 0.5 and both a hair below it")
    centre_row = np.array([[1.0, 4.0, 7.0], [2.0, 5.0, 8.0], [3.0, 6.0, 9.0]])      # sample 1 sits exactly on every feature mean
    assert duplicate_pairs([centre_row], [0, 1, 2], -0.5) == [(0, 1), (1, 2)], (
        "a sample whose scaled row is all zeros has similarity 0.0 with every sample (above r_min = -0.5 here), so it is "
        "reported with its neighbours: do not skip such a row, and do not let its division by 0 turn into NaN")
    print("exercise 1 looks right: near-copies under different donor ids are found, repeats of one donor are not")

In [ ]:
_try("exercise 1", _check_duplicate_pairs)

In [ ]:
_FLAGGED: dict = {}


def flagged_pairs(world: int, kind: str, panel: tuple = MODALITIES) -> list:
    """The pairs your `duplicate_pairs` flags at R_MIN in the layers of `panel`, for cohort `kind` of world `world`. Given.

    Comparing every pair of samples is the costly part of the tables below, so the search is run once per cohort and
    panel and remembered. The key includes your function itself: after you edit exercise 1 and run its cell again the
    key is new and the pairs are found again.
    """
    key = (duplicate_pairs, world, kind, panel)
    if key not in _FLAGGED:
        c = cohort_of(world, kind)
        _FLAGGED[key] = duplicate_pairs([getattr(c, name) for name in panel], c.donor, R_MIN)
    return _FLAGGED[key]


def _uncentred_pairs(blocks: list, donor: np.ndarray, r_min: float) -> int:
    """How many pairs a cosine WITHOUT the centring and scaling would flag. Given; for the table below only."""
    flagged = np.zeros((len(donor), len(donor)), dtype=bool)
    for block in blocks:
        unit = block / np.sqrt(np.einsum("if,if->i", block, block))[:, None]
        flagged |= np.einsum("if,jf->ij", unit, unit) > r_min
    return int((np.triu(flagged, 1) & (donor[:, None] != donor[None, :])).sum())


def _threshold_table() -> None:
    dup, bal = cohort_of(0, "duplicates"), cohort_of(0, "balanced")
    n_copy = int((dup.copy_of >= 0).sum())
    print(f"world 0, duplicates cohort: the answer key says {n_copy} samples copy another sample")
    print(f"{'r_min':>6}  {'pairs flagged, duplicates cohort':>33}  {'pairs flagged, balanced cohort':>31}")
    grid = (0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99)
    counts = []
    for r in grid:
        if r == R_MIN:             # the search at R_MIN is the one every later table needs: use the remembered one
            found, quiet = len(flagged_pairs(0, "duplicates")), len(flagged_pairs(0, "balanced"))
        else:
            found = len(duplicate_pairs([dup.protein, dup.metabolite], dup.donor, r))
            quiet = len(duplicate_pairs([bal.protein, bal.metabolite], bal.donor, r))
        counts.append(found)
        print(f"{r:>6.2f}  {found:>33}  {quiet:>31}")
    hits = [k for k, found in enumerate(counts) if found == n_copy]
    if hits:
        first, last = hits[0], hits[-1]
        unbroken = hits == list(range(first, last + 1))
        print(f"the count equals the number of copies at {len(hits)} of the {len(grid)} grid values, from {grid[first]:.2f} to "
              f"{grid[last]:.2f}; " + ("every grid value between them matches" if unbroken else "some grid value between them does not")
              + f"; R_MIN = {R_MIN:.2f} " + ("is one of the values that match" if R_MIN in [grid[k] for k in hits]
                                              else "is not one of them"))
    else:
        print(f"no grid value of r_min gives exactly {n_copy} flagged pairs")
    raw = _uncentred_pairs([dup.protein, dup.metabolite], dup.donor, R_MIN)
    print(f"the same cosine WITHOUT centring and scaling each feature flags {raw} pairs at R_MIN, out of "
          f"{len(dup.y) * (len(dup.y) - 1) // 2} pairs of samples")


def _exact_copies_table() -> None:
    exact = 0
    for w in range(N_WORLDS):
        c = cohort_of(w, "duplicates")
        truth = [(int(c.copy_of[i]), int(i)) for i in np.flatnonzero(c.copy_of >= 0)]
        exact += flagged_pairs(w, "duplicates") == sorted(truth)
    print(f"over {N_WORLDS} worlds at R_MIN, the flagged pairs are exactly the pairs (original, copy) in {exact} worlds")


if _DEMOS:
    _try("the similarity threshold", _threshold_table, needs=("exercise 1",))
    _try("the copies at the chosen threshold", _exact_copies_table, needs=("exercise 1",))

## 4. Exercise 2 — `merge_donors()`: from flagged pairs to one real donor

A flagged pair says that two samples came from the same material, so their two donor ids are one person. Links
chain: if donor 7 is linked to donor 12 and donor 12 to donor 31, then 7, 12 and 31 are one person, although no
pair links 7 to 31. `merge_donors(donor, pairs)` takes the manifest donor id of every sample and the flagged pairs
(sample indices, as exercise 1 returns them) and returns an integer array of the same length: for each sample,
the **smallest donor id** among all the donors linked to its own, directly or through a chain. A donor that is
not linked to anything keeps its id, and a pair whose two samples share a donor id changes nothing. The order
of the pairs must not matter, and the arrays you are given must not be changed. `group_folds` on the result
then keeps real people whole.

<details><summary>💡 Hint 1 — what to think about</summary>

If the link 7–12 arrives first and 12–31 second, what has to happen to the label of everything already linked to 12?
What if the two links arrive in the other order? Count how many different donors a chain can have.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Think of donor ids as nodes and flagged pairs as edges, and give every connected group of nodes one name: its
smallest id. Either keep a table from each id to the smallest id it is known to reach and, when a pair joins two
groups, point the group with the larger name at the other one; or walk each group once. Whatever you loop on,
make sure the loop must end, even when the links form a cycle.
</details>

In [ ]:
def merge_donors(donor: np.ndarray, pairs: list) -> np.ndarray:
    """One real-donor id per sample: link the donors of every flagged pair, and name each chain by its smallest id.

    `donor` is the manifest donor id of every sample (integers); `pairs` is a list of (i, j) sample indices.
    Return an integer array with the shape of `donor`. A chain of links is followed all the way (7-12 and 12-31
    put 7, 12 and 31 in one group). A pair inside one donor changes nothing. Do not change the inputs.

    Example:
        >>> merge_donors(np.array([5, 5, 9, 9, 20]), [(1, 2)]).tolist()      # donors 5 and 9 are one person
        [5, 5, 5, 5, 20]
        >>> merge_donors(np.array([5, 9, 20]), [(0, 1), (1, 2)]).tolist()    # a chain 5-9-20
        [5, 5, 5]

    Returns:
        An integer array, the same length as `donor`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_merge_donors() -> None:
    donor = np.array([0, 0, 1, 1, 2, 2, 3, 4])
    pairs = [(1, 2), (3, 4), (7, 6)]
    kept, pairs_kept = donor.copy(), list(pairs)
    got = merge_donors(donor, pairs)
    assert isinstance(got, np.ndarray), f"got a {type(got).__name__}: return a numpy array (one donor id per sample), not a list"
    assert pairs == pairs_kept, "do not change `pairs` (no sorting, reversing or popping of the list you are given)"
    assert got.tolist() == [0, 0, 0, 0, 0, 0, 3, 3], (
        f"expected [0, 0, 0, 0, 0, 0, 3, 3], got {got.tolist()}. Donors 0-1 and 1-2 are linked, so donors 0, 1 and 2 are "
        "ONE person (a chain), named by the smallest id, 0; donors 3 and 4 are linked too. A merge that only follows "
        "direct links leaves donor 2 on its own")
    assert np.array_equal(donor, kept) and np.issubdtype(got.dtype, np.integer), (
        "return a new integer array and do not change `donor`")
    assert merge_donors(donor, pairs[::-1]).tolist() == got.tolist(), "the order of the pairs must not change the result"
    assert merge_donors(donor, []).tolist() == donor.tolist() and merge_donors(donor, [(0, 1)]).tolist() == donor.tolist(), (
        "with no pairs, or a pair inside one donor, every sample keeps its donor id")
    assert merge_donors(np.array([50, 50, 7, 7, 31]), [(1, 2), (3, 4)]).tolist() == [7] * 5, (
        "donor ids need not be 0, 1, 2, ...; the smallest id of the chain (7) names it")
    assert merge_donors(np.array([4, 9, 6]), [(0, 1), (1, 2), (2, 0)]).tolist() == [4, 4, 4], (
        "links that form a cycle must still end, and give the smallest id")
    assert merge_donors(np.array([-5, 10 ** 12, 3, 3]), [(0, 1), (1, 2)]).tolist() == [-5, -5, -5, -5], (
        "donor ids can be any integers (negative, or huge): key a dict by the id, do not use the id as a position in an array")
    print("exercise 2 looks right: chains of linked donors become one person")

In [ ]:
_try("exercise 2", _check_merge_donors)

In [ ]:
def sides_shared(identity: np.ndarray, folds: list) -> int:
    """Over all folds, how many real donors have samples both in the training and in the test samples. Given."""
    return sum(len(np.intersect1d(identity[train], identity[test])) for train, test in folds)


def merged_units(world: int, kind: str) -> np.ndarray:
    """The donor ids of cohort `kind` of world `world` after the donors that share a flagged pair are merged (both layers, R_MIN). Given."""
    return merge_donors(cohort_of(world, kind).donor, flagged_pairs(world, kind))


def _merge_table() -> None:
    kept, rows = [], []
    for w in range(N_WORLDS):
        c = cohort_of(w, "duplicates")
        X = panel_matrix(c, MODALITIES)
        merged = merged_units(w, "duplicates")
        same = len(set(zip(merged.tolist(), c.identity.tolist()))) == len(set(merged.tolist())) == len(set(c.identity.tolist()))
        kept.append(same)
        by_manifest, by_merged = group_folds(c.donor, CV_FOLDS, w), group_folds(merged, CV_FOLDS, w)
        rows.append((sides_shared(c.identity, by_manifest), sides_shared(c.identity, by_merged),
                     protocol_auc("C", w), cv_auc(X, c.y, by_merged), protocol_auc("D", w)))
    a = np.array(rows)
    print(f"merged units are exactly the real donors in {sum(kept)} of {N_WORLDS} worlds")
    print(f"real donors with samples on both sides of a fold, summed over the {CV_FOLDS} folds, mean over worlds: "
          f"folds by manifest donor {a[:, 0].mean():.1f}, folds by merged unit {a[:, 1].mean():.1f}")
    print(f"cross-validated AUC, mean over worlds (standard error): by manifest donor {a[:, 2].mean():.3f} ({_se(a[:, 2]):.3f}), "
          f"by merged unit {a[:, 3].mean():.3f} ({_se(a[:, 3]):.3f}), by real donor (answer key) {a[:, 4].mean():.3f} ({_se(a[:, 4]):.3f})")
    back = a[:, 2] - a[:, 3]
    print(f"what merging took back: {_signed(back.mean())} AUC (standard error {_se(back):.3f}); "
          f"merged minus real-donor folds: {_signed((a[:, 3] - a[:, 4]).mean())}")


if _DEMOS:
    _try("merging the flagged donors", _merge_table, needs=("exercise 1", "exercise 2"))

## 5. Exercise 3 — `batch_case_share()`: how many cases are on each plate?

If the plate predicts the outcome, a biomarker model can read the plate: Leek and colleagues call batch effects
that are correlated with an outcome of interest a major problem (`claims.yaml`). The first thing to look at is the
share of cases on each plate. Count **donors**, not samples. The outcome belongs to a donor; a donor with four samples
is still one donor; and here cases give more samples than controls, so a count of samples is pulled towards the
cases and no longer says how many donors of each kind a plate holds (the cell below prints both).

Every distinct pair (batch, donor) counts once, in that batch, with the label of its first sample; a donor with
samples on two plates counts once on each. Return `(batches, shares)`: the batch ids in increasing order, and for
each the share of the donors in it that are cases, as floats between 0 and 1. A range of shares says how
unbalanced the plates are, not whether that is more than chance would do; exercise 4 supplies the chance.

<details><summary>💡 Hint 1 — what to think about</summary>

How many rows should a donor with four samples on one plate contribute? What do you have to remove before you
take a mean? Which `np.unique` option gives you one position per distinct combination?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Find the first sample of every distinct (batch, donor) combination, keep only those samples' batch and label,
then for each batch id in increasing order take the mean of the labels. A batch with no case has share 0.0.
</details>

In [ ]:
def batch_case_share(batch: np.ndarray, donor: np.ndarray, y: np.ndarray) -> tuple:
    """Share of case DONORS in every batch: each distinct (batch, donor) pair counts once.

    `batch`, `donor` (integer ids) and `y` (1 for a case, 0 for a control) have one entry per sample; all of a
    donor's samples carry the donor's label. Return `(batches, shares)`: the sorted batch ids, and the fraction
    of the donors in each batch that are cases.

    Example:
        >>> batch = np.array([1, 1, 1, 1, 2, 2, 2])
        >>> donor = np.array([10, 10, 10, 11, 12, 13, 14])
        >>> y = np.array([1, 1, 1, 0, 0, 1, 0])
        >>> batches, shares = batch_case_share(batch, donor, y)    # batch 1 has 3 case SAMPLES out of 4, but 1 case donor of 2
        >>> batches.tolist(), shares.round(3).tolist()
        ([1, 2], [0.5, 0.333])

    Returns:
        A tuple of two arrays: the batch ids and the case share of each.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_batch_case_share() -> None:
    batch = np.array([1, 1, 1, 1, 2, 2, 2])
    donor = np.array([10, 10, 10, 11, 12, 13, 14])
    y = np.array([1, 1, 1, 0, 0, 1, 0])
    plates, shares = batch_case_share(batch, donor, y)
    assert isinstance(plates, np.ndarray) and isinstance(shares, np.ndarray), (
        "return two numpy arrays (the batch ids and the case shares), not lists")
    assert plates.tolist() == [1, 2] and np.allclose(shares, [0.5, 1 / 3], rtol=0, atol=1e-12), (
        f"expected batches [1, 2] and shares [0.5, 0.3333], got {plates.tolist()} and {np.round(shares, 4).tolist()}. "
        "Count DONORS: donor 10 has three samples on plate 1 but is one case donor, so plate 1 has one case donor "
        "of two (0.5), not three case samples of four (0.75)")
    plates, shares = batch_case_share(np.array([7, 7, 3, 3, 3, 9]), np.array([1, 1, 2, 3, 3, 4]), np.array([0, 0, 1, 0, 0, 1]))
    assert plates.tolist() == [3, 7, 9] and shares.tolist() == [0.5, 0.0, 1.0], (
        f"expected batches [3, 7, 9] with shares [0.5, 0.0, 1.0], got {plates.tolist()} and {shares.tolist()}. "
        "Batch ids need not be 0, 1, 2, ...: return them in increasing order; a plate with no case has share 0.0, "
        "a plate with only cases 1.0")
    plates, shares = batch_case_share(np.array([1, 2]), np.array([5, 5]), np.array([1, 1]))
    assert plates.tolist() == [1, 2] and shares.tolist() == [1.0, 1.0], (
        "a donor with samples on two plates counts once on EACH plate")
    print("exercise 3 looks right: the case share of every plate, counted by donor")

In [ ]:
_try("exercise 3", _check_batch_case_share)

In [ ]:
def _share_table() -> None:
    print("share of case DONORS on each plate (in brackets: the share of case SAMPLES), world 0")
    for kind in COHORT_KINDS:
        c = cohort_of(0, kind)
        for name in MODALITIES:
            batch = getattr(c, "batch_" + name)
            plates, shares = batch_case_share(batch, c.donor, c.y)
            by_sample = [float(c.y[batch == b].mean()) for b in plates]
            cells = "  ".join(f"{s:.2f} ({t:.2f})" for s, t in zip(shares, by_sample))
            print(f"  {kind:<11} {name:<10} {cells}   range {shares.max() - shares.min():.2f}")
    print(f"range of the donor-level case share over the {N_WORLDS} worlds (smallest to largest):")
    for kind in ("balanced", "duplicates", "confounded"):
        for name in MODALITIES:
            spans = []
            for w in range(N_WORLDS):
                c = cohort_of(w, kind)
                shares = batch_case_share(getattr(c, "batch_" + name), c.donor, c.y)[1]
                spans.append(float(shares.max() - shares.min()))
            print(f"  {kind:<11} {name:<10} {min(spans):.2f} to {max(spans):.2f}")


if _DEMOS:
    _try("case shares per plate", _share_table, needs=("exercise 3",))

## 6. Exercise 4 — `permute_units()`: shuffle the labels the way the data are clustered

A **negative control** re-runs the whole protocol on labels that carry no information and asks what AUC it
reports. A protocol that scores well on shuffled labels is reading something other than biology (Ojala and
Garriga study this permutation test of a classifier, `claims.yaml`). The shuffle has to respect how the data are
clustered. All samples of a donor share one label, so shuffle labels **between units** (donors, or merged
donors), not between samples: every sample of a unit keeps one label, and the labels of the units are only
moved around. A shuffle between samples gives each donor a mix of labels, which is a different null, and the
table below shows what it hides. (As of 19 October 2026, scikit-learn 1.9.1's `permutation_test_score` permutes
the targets among all samples, or among the samples of one group if `groups` is given: between samples, not
between units; `claims.yaml`.)

With `within`, a unit swaps labels only with units that have the same value of `within` (the same plate, say), so
the pattern "this plate holds mostly cases" survives and the biology does not. Use the generator `rng` you are
given, and nothing else, for every random choice, so the same seeded generator gives the same shuffle. Return a
new numpy array; do not change `unit`, `y` or `within`.

<details><summary>💡 Hint 1 — what to think about</summary>

How many labels does a unit have? Which array has one entry per unit, which array do you shuffle, and how do you
get back to one entry per sample? `np.unique` has an option for the way back.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Read each unit's label from its first sample, shuffle those labels with `rng.permutation` (with `within`: once
per group, using only the units in that group), and give every sample the new label of its unit. Do not draw a
random number per sample, and do not use numpy's global random state.
</details>

In [ ]:
def permute_units(unit: np.ndarray, y: np.ndarray, rng: np.random.Generator, within=None) -> np.ndarray:
    """Labels shuffled between units: every sample of a unit gets one label, and the unit labels are only moved.

    `unit` has one id per sample, and `y` is constant within a unit. Without `within`, the unit labels are shuffled
    among all units. With `within` (one group id per sample, constant within a unit), a unit swaps labels only with
    units of the same group. Use only `rng` for randomness. Return a new numpy array of the dtype of `y`.

    Example:
        >>> unit = np.array([4, 4, 9, 9, 9, 2]); y = np.array([1, 1, 0, 0, 0, 1])
        >>> y_new = permute_units(unit, y, np.random.default_rng(0))
        >>> len(set(zip(unit.tolist(), y_new.tolist())))          # one label per unit
        3
        >>> sorted(y_new[np.unique(unit, return_index=True)[1]].tolist())    # the three unit labels are the old ones, moved
        [0, 1, 1]

    Returns:
        An array with the shape of `y`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_permute_units() -> None:
    unit = np.array([0, 0, 1, 1, 1, 2, 3, 3])
    y = np.array([1, 1, 0, 0, 0, 1, 0, 0])
    kept, kept_unit = y.copy(), unit.copy()
    outs = [permute_units(unit, y, np.random.default_rng(s)) for s in range(12)]
    assert np.array_equal(y, kept) and np.array_equal(unit, kept_unit), "return a new array and do not change `unit` or `y`"
    assert all(isinstance(out, np.ndarray) for out in outs), "return a numpy array, not a list: the demos index other arrays with it"
    for out in outs:
        assert out.shape == y.shape and all(len(set(out[unit == u].tolist())) == 1 for u in range(4)), (
            "every sample of a unit must get the SAME label: shuffle one label per unit, not one per sample")
        assert sorted(out[np.unique(unit, return_index=True)[1]].tolist()) == [0, 0, 1, 1], (
            "the unit labels must only be moved around: the same number of case units and control units as before")
    assert len({tuple(o.tolist()) for o in outs}) > 1 and any(not np.array_equal(o, y) for o in outs), (
        "the labels never move: different seeds must give different shuffles (and not always the input)")
    again = permute_units(unit, y, np.random.default_rng(5)), permute_units(unit, y, np.random.default_rng(5))
    assert np.array_equal(again[0], again[1]), "the same seeded generator must give the same shuffle: use `rng` only"
    ids2, labels2 = np.array([0, 0, 1, 1, 2, 3, 3, 4, 5, 5]), np.array([1, 1, 0, 0, 1, 0, 0, 1, 0, 0])
    group2 = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])
    group2_kept = group2.copy()
    first = permute_units(ids2, labels2, np.random.default_rng(5), within=group2)
    np.random.seed(7)
    np.random.random(5)
    second = permute_units(ids2, labels2, np.random.default_rng(5), within=group2)
    assert np.array_equal(first, second) and np.array_equal(group2, group2_kept), (
        "with `within` too, the same seeded generator must give the same shuffle whatever numpy's global random state is "
        "(use `rng` for every group, nothing else), and `within` must not be changed")
    weave = np.array([4, 7, 4, 9, 7, 9, 4])
    weave_y = np.array([1, 0, 1, 0, 0, 0, 1])
    for s in range(12):
        out = np.asarray(permute_units(weave, weave_y, np.random.default_rng(s)))
        assert all(len(set(out[weave == u].tolist())) == 1 for u in (4, 7, 9)), (
            "the samples of a unit need not be neighbours (a merged donor's copies sit at the end of the table): give EVERY "
            "sample of a unit its unit's label, wherever the samples are, not only the samples next to each other")
    ids = np.array([0, 0, 1, 1, 2, 2, 3, 3, 4])
    labels = np.array([1, 1, 1, 1, 0, 0, 0, 0, 1])
    plates = np.array([0, 0, 0, 0, 1, 1, 1, 1, 2])
    for s in range(6):
        assert np.array_equal(permute_units(ids, labels, np.random.default_rng(s), within=plates), labels), (
            "with `within`, a unit may only swap labels with units of its own group: here every group has one label, "
            "so nothing can change")
    for s in range(8):
        mixed = np.asarray(permute_units(np.array([0, 0, 1, 1, 2, 2]), np.array([1, 1, 0, 0, 1, 1]), np.random.default_rng(s),
                                         within=np.array([0, 0, 0, 0, 1, 1])))
        assert mixed[:2].tolist() in ([1, 1], [0, 0]) and mixed[2:4].tolist() in ([1, 1], [0, 0]) and mixed[4:].tolist() == [1, 1], (
            "with `within` too, move one label per UNIT: both samples of a unit keep the same label")
    spread = [np.flatnonzero(np.asarray(permute_units(np.arange(6), np.array([1, 1, 0, 0, 0, 0]), np.random.default_rng(s))) == 1)
              for s in range(40)]
    assert any(int(at[1] - at[0]) not in (1, 5) for at in spread), (
        "two case units that start next to each other must sometimes be separated: shuffle the unit labels with "
        "rng.permutation, not a rotation of them")
    print("exercise 4 looks right: labels are shuffled between units, optionally inside a group")

In [ ]:
_try("exercise 4", _check_permute_units)

In [ ]:
def null_mean_auc(X: np.ndarray, y: np.ndarray, folds: list, unit: np.ndarray, within, world: int,
                  n_perm: int = N_PERM_NULL) -> float:
    """Mean cross-validated AUC over `n_perm` label permutations (units shuffled, optionally inside groups). Given."""
    rng = np.random.default_rng([SEED, world, 7000])
    return float(np.mean([cv_auc(X, permute_units(unit, y, rng, within), folds) for _ in range(n_perm)]))


def alarm(null_gap: float) -> bool:
    """A negative control is an alarm when its AUC is above chance by more than NULL_MARGIN (rounded to 6 places). Given."""
    return round(float(null_gap), 6) > NULL_MARGIN


CONTROLS = (   # (what the control does, cohort flavour, panel, folds, units shuffled, shuffled within)
    ("random rows, labels shuffled between DONORS", "balanced", MODALITIES, "rows", "donor", None),
    ("random rows, labels shuffled between SAMPLES", "balanced", MODALITIES, "rows", "sample", None),
    ("donors whole, labels shuffled between donors", "balanced", MODALITIES, "donor", "donor", None),
    ("duplicates: manifest folds, shuffled between manifest donors", "duplicates", MODALITIES, "donor", "donor", None),
    ("duplicates: manifest folds, shuffled between merged units", "duplicates", MODALITIES, "donor", "merged", None),
    ("duplicates: merged folds, shuffled between merged units", "duplicates", MODALITIES, "merged", "merged", None),
    ("confounded plates: shuffled over all donors", "confounded", ("protein",), "donor", "donor", None),
    ("confounded plates: shuffled within plate", "confounded", ("protein",), "donor", "donor", "plate"),
    ("balanced plates: shuffled within plate", "balanced", ("protein",), "donor", "donor", "plate"),
)


_CONTROL_ALARMS: dict = {}


def _control_row(k: int) -> None:
    what, kind, panel, how, shuffle, within = CONTROLS[k]
    _CONTROL_ALARMS.pop(k, None)
    results = []
    for w in range(N_WORLDS):
        c = cohort_of(w, kind)
        units = {"donor": c.donor, "sample": np.arange(len(c.y))}
        if "merged" in (how, shuffle):
            units["merged"] = merged_units(w, kind)
        folds = group_folds(units["merged"], CV_FOLDS, w) if how == "merged" else folds_for(c, how, w)
        X = panel_matrix(c, panel)
        group = c.batch_protein if within == "plate" else None
        results.append((cv_auc(X, c.y, folds), null_mean_auc(X, c.y, folds, units[shuffle], group, w)))
    a = np.array(results)
    _CONTROL_ALARMS[k] = sum(alarm(null - 0.5) for null in a[:, 1])
    print(f"{k + 1}. {what:<61} {a[:, 0].mean():>6.3f} {a[:, 1].mean():>9.3f} {_CONTROL_ALARMS[k]:>3} of {N_WORLDS}")


def _control_summary() -> None:
    if len(_CONTROL_ALARMS) < len(CONTROLS):
        print("some rows of the table did not finish (the lines above say why), so there is no summary")
        return
    print(f"an alarm is a null AUC more than {NULL_MARGIN} above chance (0.5), world by world; "
          "'AUC' is the real-label score of the same protocol")
    for (i, j), what in (((0, 1), "donor leak"), ((3, 4), "duplicate samples"), ((6, 7), "batch confound")):
        print(f"{what}: rows {i + 1} and {j + 1} differ only in what is shuffled; they alarm in "
              f"{_CONTROL_ALARMS[i]} and {_CONTROL_ALARMS[j]} of {N_WORLDS} worlds")


if _DEMOS:
    _try_rows("negative controls", f"{'negative control':<64} {'AUC':>6} {'null AUC':>9} {'alarms':>7}",
              [lambda k=k: _control_row(k) for k in range(len(CONTROLS))], _control_summary,
              needs=("exercise 1", "exercise 2", "exercise 4"))

**What the table is for.** Each row is a question: *does this protocol report a score above chance when the
labels carry no biology?* The rows come in pairs that differ in one thing only, what is shuffled, and the alarm
counts of a pair are the answer to "which control can see this trap". A control that stays quiet while the real
AUC is inflated is **blind** to that trap, and its silence is not evidence that the trap is absent. The null
AUC itself need not sit exactly at chance: with plates balanced by status, shuffling inside a plate can pull a
cross-validated null below it, and the alarm rule looks only above it. The margin is measured from chance, not from the
null's own baseline, so the alarm is conservative for a mild confound: a small gap over a baseline that sits below
chance can stay under the margin. Row 3 and the last row are the controls' own controls: a harness that alarmed on a
clean cohort would be useless.

## 7. Exercise 5 — `release_gate()`: no number without a clean bill

The harness runs three checks and hands their results to a gate. A gate that is easy to talk round is no gate: when
a check fails, the AUC must not leave the function. The inputs are the protocol's cross-validated `auc`; `split_pairs`,
the number of flagged duplicate pairs whose two samples lie in different units of the folds; `batch_p`, the
permutation p-value for batch confounding; and `null_gap`, the mean AUC of the label-permuted runs minus chance.

Three reasons can fire, always listed in this order: **"duplicates"** when `split_pairs` is above 0; **"batch"** when
`batch_p` is at or below `alpha`; **"null_control"** when `null_gap`, rounded to 6 decimal places, is strictly above
`margin` (rounding first, because 0.55 - 0.5 is 0.050000000000000044 in binary floating point). Every reason that
applies is listed, not just the first. Return a dict with exactly three keys: `"released"` (True when no reason
fired), `"auc"` (the AUC exactly as given, with no rounding, when released; `None` when not) and `"reasons"` (a list). Raise `ValueError` when
`auc`, `batch_p` or `null_gap` is NaN or `split_pairs` is negative: a check that could not be computed must not read as
a pass.

<details><summary>💡 Hint 1 — what to think about</summary>

What should a caller get when two checks fail at once? What should the `"auc"` entry hold when the answer is no?
Which of the three comparisons is strict, and which is not?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Check for the unreadable inputs first. Then test the three conditions one after another and append a reason for each
that holds, in the order given. The answer is "released" exactly when the list of reasons is empty.
</details>

In [ ]:
def release_gate(auc: float, split_pairs: int, batch_p: float, null_gap: float,
                 alpha: float = BATCH_ALPHA, margin: float = NULL_MARGIN) -> dict:
    """Return the AUC only if no check fails; otherwise say which checks did.

    Reasons, in this order: "duplicates" if split_pairs > 0; "batch" if batch_p <= alpha; "null_control" if
    round(null_gap, 6) > margin. Return {"released": bool, "auc": auc or None, "reasons": [...]}. Raise ValueError if
    auc, batch_p or null_gap is NaN, or split_pairs is negative.

    Example:
        >>> release_gate(0.71, 0, 0.40, -0.02)
        {'released': True, 'auc': 0.71, 'reasons': []}
        >>> release_gate(0.93, 3, 0.004, 0.2)
        {'released': False, 'auc': None, 'reasons': ['duplicates', 'batch', 'null_control']}

    Returns:
        A dict with the keys "released", "auc" and "reasons".
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_release_gate() -> None:
    assert release_gate(0.71, 0, 0.40, -0.02) == {"released": True, "auc": 0.71, "reasons": []}, (
        "with no failing check the gate releases the AUC unchanged, with an empty list of reasons")
    got = release_gate(0.93, 3, 0.004, 0.2)
    assert got == {"released": False, "auc": None, "reasons": ["duplicates", "batch", "null_control"]}, (
        f"got {got}. When several checks fail, list ALL of them in the order duplicates, batch, null_control, "
        "and return auc None: a refused report must not hand the number on")
    assert release_gate(0.8, 1, 0.5, 0.0)["reasons"] == ["duplicates"] and release_gate(0.8, 0, 0.5, 0.0)["released"], (
        "a single flagged pair across the folds is enough (split_pairs > 0); none is fine")
    assert release_gate(0.8, 0, 0.01, 0.0)["reasons"] == ["batch"] and release_gate(0.8, 0, 0.0100001, 0.0)["released"], (
        "a batch p-value AT alpha already refuses (p <= alpha), a hair above it does not")
    assert release_gate(0.8, 0, 0.0, 0.0)["reasons"] == ["batch"], (
        "a batch p-value of exactly 0.0 is the strongest evidence of all and must refuse (0.0 is falsy: do not test `if batch_p`)")
    assert release_gate(0.8, 0, 0.5, 0.55 - 0.5)["released"] and release_gate(0.8, 0, 0.5, 0.0501)["reasons"] == ["null_control"], (
        "round the null gap to 6 places before comparing: 0.55 - 0.5 is 0.050000000000000044 and must not alarm at "
        "margin 0.05, while 0.0501 must")
    assert release_gate(0.8, 0, 0.5, 0.2, margin=0.3)["released"] and release_gate(0.8, 0, 0.02, 0.0, alpha=0.05)["reasons"] == ["batch"], (
        "honour the alpha and margin you are given instead of the defaults")
    assert release_gate(0.8, 0, 0.5, -0.4)["released"], "a null AUC far BELOW chance (a negative gap) is not an alarm"
    assert release_gate(0.0, 0, 0.5, 0.0)["auc"] == 0.0, (
        "an AUC of exactly 0.0 is a number like any other: release it (0.0 is falsy, so do not test `if auc`)")
    assert release_gate(0.7134567891234, 0, 0.5, 0.0)["auc"] == 0.7134567891234, (
        "hand on the AUC exactly as you were given it: do not round it, format it or convert it on the way out")
    nan = float("nan")
    for bad in ((nan, 0, 0.5, 0.0), (0.7, 0, nan, 0.0), (0.7, 0, 0.5, nan), (0.7, -1, 0.5, 0.0),
                (0.9, 3, 0.5, nan), (nan, 3, 0.5, 0.0), (0.7, 2, nan, 0.0), (0.9, -2, 0.001, 0.3)):
        try:
            release_gate(*bad)
        except ValueError:
            continue
        raise AssertionError(f"release_gate{bad} must raise ValueError: a NaN input, or a negative pair count, is an "
                             "unreadable check and must never read as a pass, even when another check has already failed. "
                             "Test for it first, before any reason is listed")
    print("exercise 5 looks right: the AUC leaves the gate only when every check is quiet")

In [ ]:
_try("exercise 5", _check_release_gate)

In [ ]:
def split_pairs_of(pairs: list, unit: np.ndarray) -> int:
    """How many flagged pairs have their two samples in different units, so that a fold boundary can pass between them. Given."""
    return sum(1 for i, j in pairs if unit[i] != unit[j])


def batch_p_value(batch: np.ndarray, unit: np.ndarray, y: np.ndarray, rng: np.random.Generator,
                  n_perm: int = N_PERM_BATCH) -> float:
    """Permutation p-value for "the plates differ in case share by more than chance". Given.

    The statistic is the range of the donor-level case share over the plates (exercise 3); the null shuffles the
    labels between units (exercise 4) and keeps every sample on its plate. p = (1 + shuffles at least as unbalanced
    as the data) / (1 + n_perm).
    """
    def spread(labels: np.ndarray) -> float:
        _, shares = batch_case_share(batch, unit, labels)
        return round(float(shares.max() - shares.min()), 9)
    observed = spread(y)
    at_least = sum(spread(permute_units(unit, y, rng)) >= observed for _ in range(n_perm))
    return (1 + at_least) / (1 + n_perm)


def run_harness(cohort: Cohort, panel: tuple, unit: np.ndarray, world: int, pairs: list | None = None) -> dict:
    """Audit a biomarker report for `panel` with folds that keep `unit` whole. Given; it calls exercises 1 to 5.

    Three checks, then the gate: flagged duplicate pairs that the folds can split; a batch p-value for each layer in
    the panel (the smallest counts); and a negative control that shuffles labels between units inside the cells of
    the panel's plates. The result holds what each check found and the gate's answer; the AUC is in it only if released.
    `pairs` are the pairs `duplicate_pairs` flags in the panel's layers at R_MIN, if the caller has them already; they
    are found here when it has not.
    """
    blocks = [getattr(cohort, name) for name in panel]
    X = np.hstack(blocks)
    folds = group_folds(unit, CV_FOLDS, world)
    if pairs is None:
        pairs = duplicate_pairs(blocks, cohort.donor, R_MIN)
    plates = [getattr(cohort, "batch_" + name) for name in panel]
    rng = np.random.default_rng([SEED, world, 7100])
    p_batch = min(batch_p_value(b, unit, cohort.y, rng) for b in plates)
    cells = plates[0] if len(plates) == 1 else plates[0] * 1000 + plates[1]
    gap = null_mean_auc(X, cohort.y, folds, unit, cells, world) - 0.5
    verdict = release_gate(cv_auc(X, cohort.y, folds), split_pairs_of(pairs, unit), p_batch, gap)
    return {"pairs": len(pairs), "split": split_pairs_of(pairs, unit), "batch_p": p_batch, "null_gap": gap, **verdict}


HARNESS_ROWS = (   # (cohort flavour, panel, units the folds keep whole)
    ("balanced", MODALITIES, "manifest"),
    ("duplicates", MODALITIES, "manifest"),
    ("duplicates", MODALITIES, "merged"),
    ("confounded", MODALITIES, "merged"),
    ("confounded", ("protein",), "merged"),
    ("confounded", ("metabolite",), "merged"),
    ("all three", MODALITIES, "merged"),
    ("all three", ("metabolite",), "manifest"),
    ("all three", ("metabolite",), "merged"),
)


def _should_refuse(cohort: Cohort, kind: str, panel: tuple, unit: np.ndarray) -> bool:
    """What the ANSWER KEY says: a copy split by the units, or a protein panel on a confounded cohort. Given."""
    copies = np.flatnonzero(cohort.copy_of >= 0)
    split = bool(np.any(unit[copies] != unit[cohort.copy_of[copies]]))
    return split or (COHORT_KINDS[kind][1] and "protein" in panel)


_HARNESS_AGREE: dict = {}


def _harness_row(k: int) -> None:
    kind, panel, units = HARNESS_ROWS[k]
    _HARNESS_AGREE.pop(k, None)
    refused, fired, aucs, agree = 0, np.zeros(3, dtype=int), [], 0
    for w in range(N_WORLDS):
        c = cohort_of(w, kind)
        unit = merged_units(w, kind) if units == "merged" else c.donor
        out = run_harness(c, panel, unit, w, flagged_pairs(w, kind, panel))
        refused += not out["released"]
        fired += [r in out["reasons"] for r in ("duplicates", "batch", "null_control")]
        if out["released"]:
            aucs.append(out["auc"])
        agree += (not out["released"]) == _should_refuse(c, kind, panel, unit)
    _HARNESS_AGREE[k] = agree
    shown = f"{np.mean(aucs):.3f}" if aucs else "-"
    print(f"{kind:<11} {'+'.join(panel):<18} {units:<9} {refused:>5} of {N_WORLDS}  "
          f"{fired[0]:>8} / {fired[1]:>5} / {fired[2]:>4}              {shown}")


def _harness_summary() -> None:
    if len(_HARNESS_AGREE) < len(HARNESS_ROWS):
        print("some rows of the table did not finish (the lines above say why), so there is no summary")
        return
    wrong = [f"{kind} / {'+'.join(panel)} / {units} ({_HARNESS_AGREE[k]} of {N_WORLDS})"
             for k, (kind, panel, units) in enumerate(HARNESS_ROWS) if _HARNESS_AGREE[k] < N_WORLDS]
    if wrong:
        print("rows where the harness disagreed with the answer key in some world: " + "; ".join(wrong))
    else:
        print(f"the harness agreed with the answer key in every world of every row ({len(HARNESS_ROWS)} rows, {N_WORLDS} worlds)")


if _DEMOS:
    _try_rows("the harness", f"{'cohort':<11} {'panel':<18} {'units':<9} {'refused':>8}  fired (duplicates / batch / null)  AUC when released",
              [lambda k=k: _harness_row(k) for k in range(len(HARNESS_ROWS))], _harness_summary, needs=tuple(_EXERCISES))

## 8. Common mistakes

- **Making the flag go away instead of acting on it.** Raising `r_min` until `duplicate_pairs` returns nothing hides
  the finding. The finding means two donor ids are one person: merge them, and rerun the harness with the merged
  units as the grouping.
- **Counting samples instead of donors**, for plates and for the shuffle. The outcome belongs to a donor.
- **Shuffling labels between samples** in a cohort with repeated samples. It breaks the link between a donor and its
  label, so a protocol that memorises donors looks clean.
- **Taking a quiet control for proof.** Each control is blind to some trap; the table in section 6 shows which.
- **Hoping a better split cures a confounded plate.** Leaving a plate out changes what the AUC measures; it does not
  make the cohort balanced, and the table in section 2 shows where it lands against the balanced reference. The gate
  refuses, and the fix is a new run with plates balanced by status.
- **Letting a check read the answer key.** `identity` and `copy_of` exist so this notebook can grade the checks; a
  check that reads them proves nothing about a cohort that has no key.
- **Reading "agreed with the answer key" as an error rate.** It describes these worlds. The key refuses only where a copy
  is split or a protein panel sits on confounded plates, and the batch check's false-alarm rate was not measured: the
  balanced cohorts are filled in blocks, so their plates are balanced by construction. On plates assigned at random it
  would fire no more often than `BATCH_ALPHA` for one layer, and more often for a panel of two (the smaller p-value counts).

## 9. Self-check

1. A flexible model scores a high AUC on folds of random rows and a clearly lower one on folds that keep each donor
   whole. Which reading is right for a biomarker that will be used on newly recruited donors?
   - (a) the random-row score is right: it uses more of the data for training
   - (b) the donor-grouped score is too pessimistic, because it throws samples away
   - (c) only the donor-grouped score describes new donors; the random-row score includes recognising donors
2. Two samples carry different donor ids but have almost identical profiles, far above every other pair. What should
   the harness do before any biomarker is reported?
   - (a) treat them as one donor: merge their ids and keep them in the same fold
   - (b) delete the later sample and carry on
   - (c) lower the similarity threshold until more pairs are found
3. Why does shuffling labels between samples give a null AUC near chance even for a protocol that memorises donors?
   - (a) a null AUC is always near chance
   - (b) the shuffle gives each donor mixed labels, so recognising a donor no longer predicts a label
   - (c) shuffling removes the batch effect
4. Cases have more samples than controls. For "is the plate confounded with status", which share is the right one?
   - (a) the share of case donors on each plate
   - (b) the share of case samples on each plate
   - (c) the share of samples overall

Answers come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (  # (exercise, its check)
        ("exercise 1", _check_duplicate_pairs),
        ("exercise 2", _check_merge_donors),
        ("exercise 3", _check_batch_case_share),
        ("exercise 4", _check_permute_units),
        ("exercise 5", _check_release_gate),
    )
    import contextlib
    import io
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in _ALL_CHECKS:
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

## What you built, and what it cannot see

You built the four checks and the gate that a biomarker report has to pass, and you measured them against an answer
key: a finder for samples registered twice, a merge that turns them into one person, a plate balance counted by
donor, a label shuffle that respects the clustering, and a gate that withholds the number. **What the harness cannot
see:** a person enrolled twice, who shows up at donor-level similarity and not at copy-level (the threshold ignores
it on purpose); a batch effect tied to something that was never recorded; a model that leaks in a way none of these
controls shuffles; how often the batch check raises a false alarm on plates assigned at random (these balanced cohorts
are filled in blocks, so that was not measured); and anything about a real cohort, whose plates are messier than
these. A quiet harness is a necessary condition for a report, never a sufficient one. The capstone of this programme runs this audit before its
biomarker shortlist.

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.